# FIGARO parser walkthrough

This notebook is the practical guide for parsing FIGARO from the Eurostat API in MARIO.

## what this notebook covers

- which Eurostat dataflows MARIO uses;
- how `SUT` and `IOT` parsing differ;
- how `year=`, `countries=`, and `iot_mode=` control the request;
- why no local FIGARO path is needed anymore.

## source links

- Eurostat Statistics API: https://ec.europa.eu/eurostat/web/user-guides/data-browser/api-data-access/api-introduction
- FIGARO data browser folder: https://ec.europa.eu/eurostat/databrowser/explore/all/naio?lang=en&subtheme=naio.naio_10.naio_10_fcp&display=list&sort=category

MARIO downloads FIGARO through the Eurostat API. The old local flat-file `path` workflow is deprecated.

## dataflow groups

The parser selects the dataflow suffix from `year`:

- `2010`-`2013`: `s1`, `u1`, `ip1`, `ii1`
- `2014`-`2017`: `s2`, `u2`, `ip2`, `ii2`
- `2018`-`2021`: `s3`, `u3`, `ip3`, `ii3`
- `2022` onwards: `s4`, `u4`, `ip4`, `ii4`

For example, a 2022 SUT uses `naio_10_fcp_s4` and `naio_10_fcp_u4`.

## key arguments

- `table`: choose `"SUT"` or `"IOT"`.
- `year`: required.
- `countries`: optional list of FIGARO country codes. Use it while developing to avoid downloading the full table.
- `iot_mode`: only for `table="IOT"`; use `"auto"`, `"product"`, or `"industry"`.
- `unit`: currently `"MIO_EUR"`.

In [ ]:
import mario

## parse a FIGARO SUT

The example below limits the request to two countries. Remove `countries=` to parse the full FIGARO table.

In [ ]:
db_sut = mario.parse_figaro(
    table="SUT",
    year=2022,
    countries=["IT", "ME"],
    calc_all=False,
)

db_sut

## parse a FIGARO IOT

For `IOT` parsing, `iot_mode="auto"` defaults to the product-by-product dataflow.

In [ ]:
db_iot = mario.parse_figaro(
    table="IOT",
    year=2022,
    iot_mode="product",
    countries=["IT", "ME"],
    calc_all=False,
)

db_iot

## first inspection

Once parsed, the result is a standard MARIO database.

In [ ]:
db_sut.get_index("Region")[:5]
db_sut.get_index("Activity")[:5]
db_sut.get_index("Commodity")[:5]

## practical warnings

- Full FIGARO tables are large; use `countries=` while developing workflows.
- MARIO chunks Eurostat API requests by `c_orig` internally to avoid extraction-size limits.
- The old local `path` argument is deprecated and ignored.